# Chapter 3: Methodology - Data Preprocessing

## 3.1 Introduction
Raw genomic (VCF) and transcriptomic (GCT) data must be transformed into numerical formats suitable for machine learning algorithms. This notebook details the feature engineering pipeline.

## 3.2 Parsing VCF to Genotype Matrix
We convert the Variant Call Format (VCF) into a genotype matrix where rows represent samples and columns represent genetic variants (SNPs).
Genotypes are encoded as:
- 0: Homozygous Reference (Ref/Ref)
- 1: Heterozygous (Ref/Alt)
- 2: Homozygous Alternate (Alt/Alt)


In [ ]:
import pandas as pd
import numpy as np
import glob
import os

# Handle optional pysam dependency
try:
    import pysam
    PYSAM_AVAILABLE = True
except ImportError:
    PYSAM_AVAILABLE = False
    print("pysam not installed. Using manual VCF parsing (slower but compatible with Windows).")

def parse_vcf_manual(vcf_path):
    """
    Manually parses VCF file without pysam.
    """
    variant_ids = []
    genotypes = []
    samples = []
    
    with open(vcf_path, 'r') as f:
        for line in f:
            if line.startswith('##'):
                continue
            if line.startswith('#CHROM'):
                samples = line.strip().split('\t')[9:]
                continue
            
            # Parse data line
            parts = line.strip().split('\t')
            chrom, pos, ref, alt = parts[0], parts[1], parts[3], parts[4]
            format_str = parts[8]
            sample_data = parts[9:]
            
            var_id = f"{chrom}_{pos}_{ref}_{alt}"
            variant_ids.append(var_id)
            
            row_gt = []
            for s in sample_data:
                # GT is usually the first field, e.g., 0|0:...
                gt_str = s.split(':')[0]
                if '.' in gt_str:
                    row_gt.append(np.nan)
                else:
                    # Sum alleles: 0|1 -> 1, 1|1 -> 2
                    alleles = gt_str.replace('|', '/').split('/')
                    row_gt.append(sum(int(a) for a in alleles if a.isdigit()))
            genotypes.append(row_gt)
            
    df_gt = pd.DataFrame(genotypes, columns=samples, index=variant_ids).T
    return df_gt

def vcf_to_dataframe(vcf_path):
    if PYSAM_AVAILABLE:
        try:
            vcf_in = pysam.VariantFile(vcf_path)
            samples = list(vcf_in.header.samples)
            variant_ids = []
            genotypes = []
            for record in vcf_in:
                var_id = f"{record.chrom}_{record.pos}_{record.ref}_{','.join(record.alts)}"
                variant_ids.append(var_id)
                row_gt = []
                for sample in samples:
                    gt = record.samples[sample]['GT']
                    if gt is None or None in gt:
                        row_gt.append(np.nan)
                    else:
                        row_gt.append(sum(gt))
                genotypes.append(row_gt)
            df_gt = pd.DataFrame(genotypes, columns=samples, index=variant_ids).T
            return df_gt
        except Exception as e:
            print(f"pysam failed ({e}), falling back to manual parsing.")
            return parse_vcf_manual(vcf_path)
    else:
        return parse_vcf_manual(vcf_path)

# Test on one buffer
# df = vcf_to_dataframe('../data/raw/MTHFR_chr1.vcf')
# print(df.head())

## 3.3 Processing GTEx Transcriptomic Data
We filter the GTEx data to focus on neural tissues.

In [ ]:
def filter_neural_tissues(gct_path):
    # ... (Same logic as before, just saving output)
    if not os.path.exists(gct_path):
        print("GTEx file not found. Skipping expression processing.")
        return pd.DataFrame()
        
    try:
        # Read chunks or minimal cols if large. For now read full.
        df = pd.read_csv(gct_path, sep='\t', skiprows=2)
        neural_cols = [c for c in df.columns if 'Brain' in c or 'Spinal' in c]
        # For synthetic targets we need numerical data
        return df[neural_cols].T # Samples as rows
    except Exception as e:
        print(f"Error: {e}")
        return pd.DataFrame()

# df_expr = filter_neural_tissues('../data/raw/GTEx_median_tpm.gct.gz')

## 3.4 Save Processed Data
Save the matrices for Model Development.

In [ ]:
def run_pipeline():
    vcfs = glob.glob('../data/raw/*.vcf')
    if not vcfs:
        print("No VCF files found. Run Data Acquisition first.")
        return
    
    # Process VCFs
    print("Processing VCFs...")
    dfs = []
    for vcf in vcfs:
        print(f"  Parsing {vcf}...")
        df = vcf_to_dataframe(vcf)
        dfs.append(df)
    
    if dfs:
        full_genotype_df = pd.concat(dfs, axis=1)
        full_genotype_df.fillna(0, inplace=True)
        full_genotype_df.to_csv('../data/processed/genotype_matrix.csv')
        print(f"Saved Genotype Matrix: {full_genotype_df.shape}")
        
        # Create Mock Targets for these samples to allow End-to-End testing
        n_samples = len(full_genotype_df)
        print("Creating Mock Targets for real samples...")
        targets = np.random.randint(0, 2, size=n_samples)
        pd.DataFrame(targets, index=full_genotype_df.index, columns=['Target']).to_csv('../data/processed/targets.csv')
        
    # Process Expression
    # (Skipping for now as we might not have downloaded the 4GB GCT file)
    # This ensures the pipeline doesn't break if you didn't define fetch_gtex_median_expression
    pd.DataFrame().to_csv('../data/processed/expression_matrix.csv')

run_pipeline()